In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler,MinMaxScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.decomposition import PCA

In [ ]:
df = pd.read_csv("01 covid_toy - 01 covid_toy.csv")
display(df.head())

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [ ]:
print(df.dtypes)

age            int64
gender        object
fever        float64
cough         object
city          object
has_covid     object
dtype: object


In [ ]:
numeric_columns = df.select_dtypes(include=np.number).columns

print("Numeric Columns:")
print(numeric_columns)

Numeric Columns:
Index(['age', 'fever'], dtype='object')


In [ ]:
categorical_columns = df.select_dtypes(include="object").columns

print("categorical Columns:")
print(categorical_columns)

categorical Columns:
Index(['gender', 'cough', 'city', 'has_covid'], dtype='object')


In [ ]:
df["city"] = df["city"].astype("object")

In [ ]:
print(df.isnull().sum())

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64


In [ ]:
df=df.replace("?",np.nan)
df.isnull().sum()

,0
age,0
gender,0
fever,10
cough,0
city,0
has_covid,0


In [ ]:
df['fever'] = pd.to_numeric(df['fever'])

In [ ]:
df['fever'] = df['fever'].fillna(df['fever'].mean())

In [ ]:
print(df.isnull().sum())

age          0
gender       0
fever        0
cough        0
city         0
has_covid    0
dtype: int64


In [ ]:
scaler=MinMaxScaler().set_output(transform="pandas")
scaled_df=scaler.fit_transform(df[numeric_columns])
scaled_df.head()

,age,fever
0,0.696203,0.833333
1,0.278481,0.333333
2,0.468354,0.500000
3,0.329114,0.000000
4,0.759494,0.500000


In [ ]:
categorical_columns=["cough","has_covid"]

In [ ]:
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
).set_output(transform="pandas")

encoder_data = encoder.fit_transform(df[categorical_columns])

display(encoder_data.head())

,cough_Bangalore,cough_Delhi,cough_Kolkata,cough_Mumbai,has_covid_No,has_covid_Yes
0,0.0,0.0,1.0,0.0,1.0,0.0
1,0.0,1.0,0.0,0.0,0.0,1.0
2,0.0,1.0,0.0,0.0,1.0,0.0
3,0.0,0.0,1.0,0.0,1.0,0.0
4,0.0,0.0,0.0,1.0,1.0,0.0


In [ ]:
X = df.drop("has_covid",axis=1)
y=df["has_covid"]

In [ ]:
from sklearn.feature_selection import f_classif

# Encode the categorical target variable y to numeric (0 and 1)
y_numeric = y.map({'No': 0, 'Yes': 1})

X_numeric = X.select_dtypes(include=np.number)

# Since there are only 2 numeric features, we set k=2
selector = SelectKBest(
    score_func=f_classif,
    k=2
).set_output(transform='pandas')

X_selected = selector.fit_transform(X_numeric, y_numeric)

print(X_selected.head())

   age  fever
0   60  103.0
1   27  100.0
2   42  101.0
3   31   98.0
4   65  101.0
